# Aperture photometry

`author` Julien Morin <julien.morin@umontpellier.fr>

`date` 11 Oct 2023

References: *Handbook of CCD Astronomy* §5.4 ;
__[Photutils documentation #Getting Started with Photutils](https://photutils.readthedocs.io/en/stable/getting_started.html)__ ;
__[Photutils documentation #Aperture Photometry](https://photutils.readthedocs.io/en/stable/aperture.html)__

* load and check calibrated science frame
* find sources
* define photometric apertures for sources and sky
* perform aperture photometry

In [ ]:
# import and settings
%matplotlib notebook
import os
import numpy as np
import matplotlib.pyplot as plt
import astropy.io.fits as pf
from astropy.stats import mad_std, sigma_clipped_stats
from astropy import units as u
from astropy.visualization import ZScaleInterval
import ccdproc as ccdp
from photutils.aperture import aperture_photometry, CircularAperture, CircularAnnulus
from photutils.detection import DAOStarFinder

## Load calibrated science data and check statistics

In [ ]:
path_sci = "processed/Wasp_33-0009_bf.fits"
ccd_sci = ccdp.CCDData.read(path_sci)
ccd_sci.to_hdu().info()

In [ ]:
### ToDo: check the data unit

In [ ]:
# For simplicity we only focus on a 750x750 pix section of the image
ccd_sci = ccd_sci[0:750,1000:1750]
#
plt.figure()
interval = ZScaleInterval()
z1, z2 = interval.get_limits(ccd_sci.data)
plt.imshow(ccd_sci, origin='lower', vmin=z1, vmax=z2)
plt.show()

In [ ]:
### ToDo: print statistics and check background level

## Find sources

In [ ]:
FWHM = 5. # ToDo: set the full width at half-maximum (in pixels) of the PSF according to your image
bkg_sigma = mad_std(ccd_sci)
print("Background standard deviation: %.3f" % bkg_sigma)
daofind = DAOStarFinder(fwhm=FWHM, threshold=7.*bkg_sigma)
sources = daofind(ccd_sci.data)
print("%d sources found" % len(sources))
print(type(sources))
sources.sort(keys='peak')
for col in sources.colnames:
    sources[col].info.format = '%.8g'
print(sources[::-1])

## Define photometric apertures

In [ ]:
# Define circular aperture around the star and an annular aperture to compute the sky background level
positions = np.transpose((sources['xcentroid'], sources['ycentroid']))
star_apertures = CircularAperture(positions, r=2.*FWHM)
sky_apertures = CircularAnnulus(positions, r_in=5.*FWHM, r_out=7.*FWHM)

In [ ]:
# Overplot photometric apertures
plt.figure()
plt.imshow(ccd_sci, origin='lower', vmin=z1, vmax=z2)
star_apertures.plot(color='red', lw=1.5, alpha=0.5)
sky_apertures.plot(color='white', lw=1.5, alpha=0.5)
plt.show()

In [ ]:
# Plot star and sky apertures for source with index 'istar'
istar = 0
mask_star = star_apertures[istar].to_mask(method='center')
ap_star = mask_star.multiply(ccd_sci.data)
mask_sky = sky_apertures[istar].to_mask(method='center')
ap_sky = mask_sky.multiply(ccd_sci.data)
#
fig = plt.figure(figsize=(9,4))
#
ax1 = fig.add_subplot(131)
interval1 = ZScaleInterval()
z1, z2 = interval1.get_limits(ap_star)
im1 = ax1.imshow(ap_star.data, origin='lower', vmin=z1, vmax=z2, interpolation='nearest')
fig.colorbar(im1)
ax1.set_title('star aperture #%d' % istar)
# Show background w/ same LUT as the star
ax2 = fig.add_subplot(132)
im2 = ax2.imshow(ap_sky.data, origin='lower', vmin=z1, vmax=z2, interpolation='nearest')
fig.colorbar(im2)
ax2.set_title('sky aperture #%d' % istar)
# Show background w/ its own LUT
ax3 = fig.add_subplot(133)
interval2 = ZScaleInterval()
z1, z2 = interval2.get_limits(ap_sky)
im3 = ax3.imshow(ap_sky.data, origin='lower', vmin=z1, vmax=z2, interpolation='nearest')
fig.colorbar(im3)
ax3.set_title('sky aperture #%d' % istar)
#
plt.show()

## Compute photometry

In [ ]:
# perform aperture photometry on star and sky apertures
print("star aperture area: ", star_apertures.area)
print("sky aperture area : ", sky_apertures.area)
apers = [star_apertures, sky_apertures]
phot_table = aperture_photometry(ccd_sci.data, apers, error=ccd_sci.uncertainty.array, method='exact')
phot_table.rename_column('aperture_sum_0', 'star_sum')
phot_table.rename_column('aperture_sum_err_0', 'star_err')
phot_table.rename_column('aperture_sum_1', 'sky_sum')
phot_table.rename_column('aperture_sum_err_1', 'sky_err')
sky_masks = sky_apertures.to_mask(method='exact')
for col in phot_table.colnames:
    phot_table[col].info.format = '%.5g'
print(phot_table[::-1])

In [ ]:
### Further work: 
###   - use the sky annulus measurement to correct stellar photometry
###   - experiment w/ different radii and plot recovered stellar and sky flux as a function of r, r_in, r_out
###   - compute uncertainties manually and compare w/ aperture_photometry result, identify dominant noise sources